# HDP00941 — ASCENT Pilot 1 Survey: Secondary Analysis

**Study:** ASCENT (Addressing Symptoms and Cancer-related needs through Equitable Navigation and Technology)  
**Data source:** `ascent_pilot_1_survey_responses.xlsx` (Stata `.dta` format)  
**Analysis type:** Secondary/descriptive analysis of Pilot 1 patient-reported survey outcomes  

---

### Contents
1. Setup & Data Load  
2. Sample Description  
3. Care Team Engagement  
4. Support Types Received  
5. Communication & Access  
6. Self-Efficacy Outcomes  
7. Resource Utilization  
8. Materials Comprehension & Helpfulness  
9. Overall Satisfaction & Expectations  
10. Summary Table  
11. Heatmap — Outcome Scores by Subgroup  

## 1. Setup & Data Load

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
import pyreadstat
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.1f}'.format)

# ── Palette ──────────────────────────────────────────────────────────────────
BLUE   = '#2563EB'
TEAL   = '#0D9488'
AMBER  = '#D97706'
ROSE   = '#E11D48'
SLATE  = '#475569'
COLORS = [BLUE, TEAL, AMBER, ROSE, SLATE, '#7C3AED', '#059669', '#DC2626']
sns.set_theme(style='whitegrid', font_scale=1.05)

DATA_PATH = 'ascent_pilot_1_survey_responses.xlsx'  # Stata .dta — update path if needed

df_all, meta = pyreadstat.read_dta(DATA_PATH)
VL = meta.variable_value_labels  # shorthand for value labels

print(f'Total records loaded : {len(df_all)}')

## 2. Sample Description

All 51 records span two pilot cohorts. The primary survey analysis is restricted to participants who **completed the REDCap survey** (`redcap_yn == 1`). All 35 survey completers attended all three ASCENT visits (intake, planning, final).

In [ ]:
# ── Define analytic sub-samples ───────────────────────────────────────────────
df_p1   = df_all[df_all['pilot'] == 1].copy()          # Pilot 1 only (n=42)
survey  = df_all[df_all['redcap_yn'] == 1].copy()      # Survey completers (n=35)

print('=== SAMPLE FLOW ===')
print(f'  Total enrolled (Pilot 1 + 2)  : {len(df_all)}')
print(f'  Pilot 1 only                  : {len(df_p1)}')
print(f'  Pilot 2 only                  : {len(df_all[df_all["pilot"]==2])}')
print(f'  Withdrawn                     : {int(df_all["withdrawn"].sum())}')
print(f'  Completed REDCap survey       : {len(survey)}')
print(f'  Completed research interview  : {int(df_all["interview_yn"].sum())}')

In [ ]:
# ── Demographics of survey completers ─────────────────────────────────────────
def pct(series, val):
    n = series.notna().sum()
    c = (series == val).sum()
    return c, n, 100*c/n if n else 0

rows = []
# Age
rows.append(('Age (mean ± SD)', f"{survey['age_at_consent'].mean():.1f} ± {survey['age_at_consent'].std():.1f}", ''))
rows.append(('Age (range)',     f"{survey['age_at_consent'].min():.0f}–{survey['age_at_consent'].max():.0f}", ''))
# Sex
c,n,p = pct(survey['sex_numerical'], 1)
rows.append(('Female', f'{c}', f'{p:.1f}%'))
c,n,p = pct(survey['sex_numerical'], 0)
rows.append(('Male', f'{c}', f'{p:.1f}%'))
# Race
c,n,p = pct(survey['race_cat'], 0)
rows.append(('White', f'{c}', f'{p:.1f}%'))
c,n,p = pct(survey['race_cat'], 1)
rows.append(('Black or African American', f'{c}', f'{p:.1f}%'))
# Ethnicity
c,n,p = pct(survey['ethnicity_hispanic'], 1)
rows.append(('Hispanic / Latino', f'{c}', f'{p:.1f}%'))
# Language
c,n,p = pct(survey['preflanguage_num'], 2)
rows.append(('Preferred language: Spanish', f'{c}', f'{p:.1f}%'))
# Rurality
for val, lbl in VL['rucacat'].items():
    c,n,p = pct(survey['rucacat'], val)
    rows.append((f'RUCA: {lbl}', f'{c}', f'{p:.1f}%'))
# Qualifying feature
for val, lbl in VL['qualifyingfeature_num'].items():
    c,n,p = pct(survey['qualifyingfeature_num'], val)
    rows.append((f'Qualifying feature: {lbl}', f'{c}', f'{p:.1f}%'))
# Cancer category
for val, lbl in VL['cancer_category'].items():
    c,n,p = pct(survey['cancer_category'], val)
    if c > 0:
        rows.append((f'Cancer: {lbl}', f'{c}', f'{p:.1f}%'))

dem_df = pd.DataFrame(rows, columns=['Characteristic', 'N', '%'])
dem_df.index = range(1, len(dem_df)+1)
print(f'\nTable 1. Demographics of Survey Completers (N={len(survey)})')
display(dem_df)

In [ ]:
# ── Demographics bar chart ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Survey Completers — Key Demographics (N=35)', fontsize=13, fontweight='bold', y=1.01)

# Sex
sex_counts = survey['sex_numerical'].map(VL['sex_numerical']).value_counts()
axes[0].bar(sex_counts.index, sex_counts.values, color=[BLUE, TEAL])
axes[0].set_title('Sex'); axes[0].set_ylabel('Count')
for i, v in enumerate(sex_counts.values):
    axes[0].text(i, v+0.3, str(v), ha='center', fontweight='bold')

# Qualifying feature
qf = survey['qualifyingfeature_num'].map(VL['qualifyingfeature_num']).value_counts()
axes[1].bar(range(len(qf)), qf.values, color=COLORS[:len(qf)])
axes[1].set_xticks(range(len(qf)))
axes[1].set_xticklabels(qf.index, rotation=15, ha='right', fontsize=9)
axes[1].set_title('Qualifying Feature'); axes[1].set_ylabel('Count')
for i, v in enumerate(qf.values):
    axes[1].text(i, v+0.3, str(v), ha='center', fontweight='bold')

# Cancer category
cc = survey['cancer_category'].map(VL['cancer_category']).value_counts()
axes[2].barh(cc.index[::-1], cc.values[::-1], color=BLUE)
axes[2].set_title('Cancer Category'); axes[2].set_xlabel('Count')
for i, v in enumerate(cc.values[::-1]):
    axes[2].text(v+0.1, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('fig_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Care Team Engagement

Participants were asked which ASCENT care team member(s) they worked with.

In [ ]:
care_cols  = ['careteam1','careteam2','careteam3','careteam4']
care_labels = {
    'careteam1': 'Community Health Worker',
    'careteam2': 'Pain Care Manager',
    'careteam3': 'Social Worker',
    'careteam4': 'Other'
}
N = len(survey)

care_summary = {}
for col in care_cols:
    c = (survey[col] == 1).sum()
    care_summary[care_labels[col]] = (c, round(100*c/N, 1))

care_df = pd.DataFrame(care_summary, index=['N','%']).T
print(f'Care Team Members Worked With (N={N})')
display(care_df)

# Overall helpfulness of care team
helpful_map = VL['helpful']
helpful_counts = survey['helpful'].map(helpful_map).value_counts().reindex(
    [helpful_map[k] for k in sorted(helpful_map)]
).fillna(0)

print(f"\nOverall Helpfulness of ASCENT Care Team (N={survey['helpful'].notna().sum()})")
for lbl, cnt in helpful_counts.items():
    pct_val = 100*cnt/survey['helpful'].notna().sum()
    print(f"  {lbl:<30}: {int(cnt):>3}  ({pct_val:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Care team bar
members = list(care_summary.keys())
pcts    = [v[1] for v in care_summary.values()]
axes[0].barh(members[::-1], pcts[::-1], color=TEAL)
axes[0].set_xlabel('% of Survey Completers')
axes[0].set_title('Care Team Members Worked With')
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter())
for i, (p, n_) in enumerate(zip(pcts[::-1], [v[0] for v in care_summary.values()][::-1])):
    axes[0].text(p+0.5, i, f'{int(n_)} ({p}%)', va='center', fontsize=9)

# Helpfulness
h_n   = survey['helpful'].notna().sum()
h_pct = helpful_counts / h_n * 100
bars = axes[1].bar(range(len(helpful_counts)), h_pct.values,
                   color=[ROSE, AMBER, TEAL, BLUE, '#7C3AED'])
axes[1].set_xticks(range(len(helpful_counts)))
axes[1].set_xticklabels(
    [lbl.replace(' ', '\n') for lbl in h_pct.index],
    fontsize=8
)
axes[1].set_ylabel('% of Respondents')
axes[1].set_title('Overall Helpfulness of ASCENT Care Team')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, h_pct.values):
    if val > 0:
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                     f'{val:.0f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_careteam.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nMedian helpfulness: {survey["helpful"].median():.0f}/5  |  Mean: {survey["helpful"].mean():.2f}/5')

## 4. Support Types Received

In [ ]:
support_labels = {
    'support1':  'Set up medical appointments',
    'support2':  'Helped set goals',
    'support3':  'Transportation resources',
    'support4':  'Medical cost resources',
    'support5':  'Pain management information',
    'support6':  'Local community resources',
    'support7':  'Addressed cancer needs',
    'support8':  'Coaching & social support',
    'support9':  'Advocacy for patient needs',
    'support10': 'Other',
}

sup_counts = {lbl: int((survey[col] == 1).sum()) for col, lbl in support_labels.items()}
sup_pcts   = {lbl: round(100*c/N, 1) for lbl, c in sup_counts.items()}

sup_df = pd.DataFrame({'N': sup_counts, '%': sup_pcts})
sup_df = sup_df.sort_values('N', ascending=False)
print(f'Types of Support Received (N={N})')
display(sup_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sup_sorted = sup_df.sort_values('%')
bars = ax.barh(sup_sorted.index, sup_sorted['%'], color=BLUE)
ax.set_xlabel('% of Survey Completers')
ax.set_title('Types of Support Received from ASCENT Care Team', fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val, n_ in zip(bars, sup_sorted['%'], sup_sorted['N']):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2,
            f'{int(n_)} ({val}%)', va='center', fontsize=9)
ax.set_xlim(0, 110)
plt.tight_layout()
plt.savefig('fig_support_types.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Communication & Access

In [ ]:
def freq_table(col, label_dict, title, N_override=None):
    s = survey[col].dropna()
    n = N_override or len(s)
    counts = s.map(label_dict).value_counts().reindex(
        [label_dict[k] for k in sorted(label_dict)]
    ).fillna(0)
    pcts = (counts / n * 100).round(1)
    df_out = pd.DataFrame({'N': counts.astype(int), '%': pcts})
    print(f'\n{title} (n={n})')
    display(df_out)
    return df_out

freq_table('communicationfrequency', VL['communicationfrequency'],
           'How often did you talk with ASCENT care team?')

freq_table('frequencysatisfied', VL['frequencysatisfied'],
           'Satisfied with communication frequency?')

freq_table('reachteam', VL['reachteam'],
           'Ease of reaching ASCENT team when needed')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Communication & Access', fontsize=13, fontweight='bold')

comm_data = [
    ('communicationfrequency', VL['communicationfrequency'], 'Communication Frequency', axes[0]),
    ('frequencysatisfied',     VL['frequencysatisfied'],     'Frequency Satisfaction',  axes[1]),
    ('reachteam',              VL['reachteam'],              'Ease of Reaching Team',   axes[2]),
]

for col, lbl_dict, title, ax in comm_data:
    s = survey[col].dropna()
    n = len(s)
    counts = s.map(lbl_dict).value_counts().reindex(
        [lbl_dict[k] for k in sorted(lbl_dict)]
    ).fillna(0)
    pcts = counts / n * 100
    bars = ax.bar(range(len(pcts)), pcts.values, color=COLORS[:len(pcts)])
    ax.set_xticks(range(len(pcts)))
    ax.set_xticklabels(
        [str(x)[:20] for x in pcts.index], rotation=30, ha='right', fontsize=7.5
    )
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('% of Respondents')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    for bar, v in zip(bars, pcts.values):
        if v > 0:
            ax.text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.0f}%',
                    ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_communication.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Self-Efficacy Outcomes

Three items assess patient confidence after the ASCENT intervention (1 = not at all confident → 5 = extremely confident).

In [ ]:
efficacy_cols = {
    'confidentgoals':      'Confident achieving goals set with ASCENT',
    'confidentresources':  'Confident finding pain-related resources',
    'confidentmanagepain': 'Confident managing pain in the future',
}

# Note: confidentgoals has a '6 = not applicable' option — recode to NaN for numeric stats
survey_eff = survey.copy()
survey_eff['confidentgoals'] = survey_eff['confidentgoals'].replace(6, np.nan)

likert_labels = {1:'Not at all confident', 2:'Somewhat confident',
                 3:'Confident', 4:'Very confident', 5:'Extremely confident'}

eff_stats = []
for col, lbl in efficacy_cols.items():
    s = survey_eff[col].dropna()
    top2 = ((s >= 4).sum() / len(s) * 100).round(1)
    eff_stats.append({
        'Item': lbl,
        'n': len(s),
        'Median': s.median(),
        'Mean (SD)': f"{s.mean():.2f} ({s.std():.2f})",
        '% Very/Extremely Confident (top-2)': f'{top2}%'
    })

eff_df = pd.DataFrame(eff_stats).set_index('Item')
print('Self-Efficacy Outcomes')
display(eff_df)

In [ ]:
# Stacked horizontal bar chart (Likert-style)
eff_pct_data = {}
for col, lbl in efficacy_cols.items():
    s = survey_eff[col].dropna()
    row = {}
    for k in range(1, 6):
        row[likert_labels[k]] = round(100*(s==k).sum()/len(s), 1)
    eff_pct_data[lbl] = row

eff_pct_df = pd.DataFrame(eff_pct_data).T
eff_colors = ['#DC2626','#F97316','#FACC15','#22C55E','#16A34A']

fig, ax = plt.subplots(figsize=(12, 4))
left = np.zeros(len(eff_pct_df))
for (col, color) in zip(eff_pct_df.columns, eff_colors):
    vals = eff_pct_df[col].values
    bars = ax.barh(eff_pct_df.index, vals, left=left, color=color, label=col)
    for bar, val, l in zip(bars, vals, left):
        if val >= 8:
            ax.text(l + val/2, bar.get_y()+bar.get_height()/2,
                    f'{val:.0f}%', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    left += vals

ax.set_xlabel('Percentage')
ax.set_title('Self-Efficacy Outcomes (1=Not at all → 5=Extremely Confident)', fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(loc='lower right', fontsize=8, ncol=2)
ax.set_xlim(0, 100)

# Wrap long labels
ax.set_yticklabels([lbl[:40] for lbl in eff_pct_df.index], fontsize=9)

plt.tight_layout()
plt.savefig('fig_selfefficacy.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Resource Utilization

In [ ]:
utilized_labels = {
    'utilized1': 'Patient portal materials',
    'utilized2': 'Mailed pain mgmt materials',
    'utilized3': 'ASCENT website',
    'utilized4': 'Follow-up phone calls',
    'utilized5': 'Action plan',
    'utilized6': 'None of the above',
}

util_counts = {lbl: int((survey[col] == 1).sum()) for col, lbl in utilized_labels.items()}
util_pcts   = {lbl: round(100*c/N, 1) for lbl, c in util_counts.items()}

util_df = pd.DataFrame({'N': util_counts, '%': util_pcts}).sort_values('N', ascending=False)
print(f'Resources Utilized (N={N}, multiple-select)')
display(util_df)

fig, ax = plt.subplots(figsize=(9, 4))
util_sorted = util_df.sort_values('%')
bars = ax.barh(util_sorted.index, util_sorted['%'], color=TEAL)
ax.set_xlabel('% of Survey Completers')
ax.set_title('ASCENT Resources Utilized', fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val, n_ in zip(bars, util_sorted['%'], util_sorted['N']):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2,
            f'{int(n_)} ({val}%)', va='center', fontsize=9)
ax.set_xlim(0, 100)
plt.tight_layout()
plt.savefig('fig_utilization.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Materials Comprehension & Helpfulness

Each resource is evaluated on two dimensions: ease of understanding (1=very difficult → 5=very easy) and helpfulness (1=not at all → 5=extremely helpful).

In [ ]:
resources = {
    ('understandmaterials', 'helpfulmaterials'): 'Patient Portal Materials',
    ('understandmail',      'helpfulmailed'):     'Mailed Materials',
    ('understandwebsite',   'helpfulwebsite'):    'ASCENT Website',
    ('understandcalls',     'helpfulcalls'):      'Follow-up Phone Calls',
    ('understandactionplan','helpfulactionplan'):  'Action Plan',
}

rows_u, rows_h = [], []
for (u_col, h_col), lbl in resources.items():

    u = survey[u_col].dropna()
    h = survey[h_col].dropna()

    rows_u.append({
        'Resource': lbl,
        'n':        len(u),
        'Median':   u.median() if len(u) else np.nan,
        'Mean (SD)':f"{u.mean():.2f} ({u.std():.2f})" if len(u) else 'N/A',
        '% Somewhat/Very Easy (4-5)': f"{100*(u>=4).sum()/len(u):.1f}%" if len(u) else 'N/A'
    })
    rows_h.append({
        'Resource': lbl,
        'n':        len(h),
        'Median':   h.median() if len(h) else np.nan,
        'Mean (SD)':f"{h.mean():.2f} ({h.std():.2f})" if len(h) else 'N/A',
        '% Very/Extremely Helpful (4-5)': f"{100*(h>=4).sum()/len(h):.1f}%" if len(h) else 'N/A'
    })

print('Table: Ease of Understanding (1=Very Difficult → 5=Very Easy)')
display(pd.DataFrame(rows_u).set_index('Resource'))
print('\nTable: Helpfulness (1=Not at all → 5=Extremely Helpful)')
display(pd.DataFrame(rows_h).set_index('Resource'))

In [ ]:
# Mean scores comparison chart
res_names = list(resources.values())
mean_u = [survey[u_col].mean() for (u_col, h_col) in resources]
mean_h = [survey[h_col].mean() for (u_col, h_col) in resources]

x = np.arange(len(res_names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - width/2, mean_u, width, label='Ease of Understanding', color=BLUE, alpha=0.85)
b2 = ax.bar(x + width/2, mean_h, width, label='Helpfulness',           color=TEAL, alpha=0.85)

ax.set_ylabel('Mean Score (1–5 scale)')
ax.set_title('Resource Comprehension & Helpfulness by Resource Type', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(res_names, rotation=15, ha='right')
ax.set_ylim(0, 5.5)
ax.axhline(y=4, color='gray', linestyle='--', linewidth=0.8, label='Score = 4 threshold')
ax.legend()

for bar in b1:
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x()+bar.get_width()/2, h+0.05, f'{h:.2f}',
                ha='center', va='bottom', fontsize=8)
for bar in b2:
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x()+bar.get_width()/2, h+0.05, f'{h:.2f}',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('fig_materials.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Overall Satisfaction & Expectations

In [ ]:
# Amount of materials
freq_table('amount', VL['amount'], 'Amount of informational materials received')

# Needs met
needs_s = survey['needs'].dropna()
needs_met = (needs_s == 1).sum()
n_needs = len(needs_s)
print(f'\nDid ASCENT meet pain management needs? (n={n_needs})')
print(f'  Yes : {needs_met} ({100*needs_met/n_needs:.1f}%)')
print(f'  No  : {n_needs - needs_met} ({100*(n_needs-needs_met)/n_needs:.1f}%)')

# Expectations
freq_table('expectations', VL['expectations'], 'Did support meet expectations?')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Overall Satisfaction & Expectations', fontsize=13, fontweight='bold')

# Amount
amt_s   = survey['amount'].dropna()
amt_map = VL['amount']
amt_c   = amt_s.map(amt_map).value_counts().reindex([amt_map[k] for k in sorted(amt_map)]).fillna(0)
wedges, texts, autotexts = axes[0].pie(
    amt_c.values, labels=None,
    autopct='%1.0f%%', colors=COLORS[:len(amt_c)], startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}
)
[at.set_fontsize(10) for at in autotexts]
axes[0].set_title('Amount of Materials Received')
axes[0].legend(amt_c.index, loc='lower center', bbox_to_anchor=(0.5,-0.25), fontsize=8, ncol=1)

# Needs met
axes[1].pie([needs_met, n_needs - needs_met], labels=['Yes','No'],
            autopct='%1.0f%%', colors=[TEAL, ROSE], startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('ASCENT Met Pain Mgmt Needs')

# Expectations
exp_s   = survey['expectations'].dropna()
exp_map = VL['expectations']
exp_c   = exp_s.map(exp_map).value_counts().reindex([exp_map[k] for k in sorted(exp_map)]).fillna(0)
wedges2, texts2, autotexts2 = axes[2].pie(
    exp_c.values, labels=None,
    autopct='%1.0f%%', colors=[ROSE, TEAL, BLUE], startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}
)
[at.set_fontsize(10) for at in autotexts2]
short_labels = ['Disappointed', 'Met expectations', 'Exceeded expectations']
axes[2].set_title('Support Met Expectations')
axes[2].legend(short_labels, loc='lower center', bbox_to_anchor=(0.5,-0.25), fontsize=8)

plt.tight_layout()
plt.savefig('fig_satisfaction.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary Table

Consolidated key metrics for reporting.

In [ ]:
summary_rows = []

# --- Care team helpfulness ---
h = survey['helpful'].dropna()
summary_rows.append(('Care Team Helpfulness', 'Mean (SD)', f"{h.mean():.2f} ({h.std():.2f})", len(h)))
summary_rows.append(('Care Team Helpfulness', '% Very/Extremely Helpful (4–5)',
                      f"{100*(h>=4).sum()/len(h):.1f}%", len(h)))

# --- Communication ---
fs = survey['frequencysatisfied'].dropna()
sat_freq = 100*(fs==1).sum()/len(fs)
summary_rows.append(('Communication', '% Satisfied with frequency', f'{sat_freq:.1f}%', len(fs)))
rt = survey['reachteam'].dropna()
reach_ok = 100*(rt>=3).sum()/len(rt)
summary_rows.append(('Communication', '% Usually/Always could reach team', f'{reach_ok:.1f}%', len(rt)))

# --- Self-efficacy ---
for col, lbl in [
    ('confidentgoals',      'Confident achieving goals'),
    ('confidentresources',  'Confident finding resources'),
    ('confidentmanagepain', 'Confident managing pain'),
]:
    s = survey_eff[col].dropna()
    summary_rows.append(('Self-Efficacy', f'{lbl} — Mean (SD)',
                          f"{s.mean():.2f} ({s.std():.2f})", len(s)))
    summary_rows.append(('Self-Efficacy', f'{lbl} — % Top-2',
                          f"{100*(s>=4).sum()/len(s):.1f}%", len(s)))

# --- Satisfaction ---
needs_s2 = survey['needs'].dropna()
summary_rows.append(('Overall Satisfaction', '% Needs met',
                      f"{100*(needs_s2==1).sum()/len(needs_s2):.1f}%", len(needs_s2)))
exp_s2 = survey['expectations'].dropna()
met_or_exceeded = 100*(exp_s2>=2).sum()/len(exp_s2)
summary_rows.append(('Overall Satisfaction', '% Met or exceeded expectations',
                      f"{met_or_exceeded:.1f}%", len(exp_s2)))

summary_df = pd.DataFrame(summary_rows, columns=['Domain','Metric','Value','n'])
summary_df.index = range(1, len(summary_df)+1)
print('ASCENT Pilot 1 — Secondary Analysis Summary')
display(summary_df)

In [ ]:
# Export summary to CSV
summary_df.to_csv('ascent_pilot1_survey_summary.csv', index=False)
print('Summary exported to: ascent_pilot1_survey_summary.csv')
print('\nFigures saved:')
for fig_name in ['fig_demographics','fig_careteam','fig_support_types',
                  'fig_communication','fig_selfefficacy','fig_utilization',
                  'fig_materials','fig_satisfaction',
                  'fig_heatmap_overall','fig_heatmap_subgroup']:
    print(f'  {fig_name}.png')

## 11. Heatmap — Outcome Scores by Subgroup

Two heatmaps are produced:
1. **Overall item-level heatmap** — mean score for every Likert-scale item across all survey completers, colour-coded from low (red) to high (green).
2. **Subgroup comparison heatmap** — mean scores for each domain broken down by sex, ethnicity, rurality, and qualifying feature.

In [ ]:
# ── 11a. Overall item-level heatmap ──────────────────────────────────────────
# All 5-point Likert items (higher = better for all)
likert_items = {
    'helpful':              'Care team helpfulness',
    'reachteam':            'Ease of reaching team',
    'confidentgoals':       'Confident: goals',
    'confidentresources':   'Confident: resources',
    'confidentmanagepain':  'Confident: manage pain',
    'understandmaterials':  'Understand: portal materials',
    'understandmail':       'Understand: mailed materials',
    'understandwebsite':    'Understand: website',
    'understandcalls':      'Understand: phone calls',
    'understandactionplan': 'Understand: action plan',
    'helpfulmaterials':     'Helpful: portal materials',
    'helpfulmailed':        'Helpful: mailed materials',
    'helpfulwebsite':       'Helpful: website',
    'helpfulcalls':         'Helpful: phone calls',
    'helpfulactionplan':    'Helpful: action plan',
}

# Build a 1-row matrix of means for the overall heatmap
survey_eff2 = survey.copy()
survey_eff2['confidentgoals'] = survey_eff2['confidentgoals'].replace(6, np.nan)

means_overall = {lbl: survey_eff2[col].mean() for col, lbl in likert_items.items()}
hm_overall = pd.DataFrame(means_overall, index=['Mean score'])

fig, ax = plt.subplots(figsize=(16, 2.2))
sns.heatmap(
    hm_overall,
    ax=ax,
    vmin=1, vmax=5,
    cmap='RdYlGn',
    annot=True,
    fmt='.2f',
    annot_kws={'size': 9},
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Mean score (1–5)', 'shrink': 0.8}
)
ax.set_title('Overall Mean Scores — All Likert-Scale Items (N=35)', fontsize=12, fontweight='bold', pad=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('fig_heatmap_overall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Higher scores = more positive outcome (scale 1–5)')

In [ ]:
# ── 11b. Subgroup comparison heatmap ─────────────────────────────────────────
# Domain-level mean scores split by key demographic subgroups

domain_cols = {
    'Care Team\nHelpfulness':   ['helpful'],
    'Ease of\nReach':           ['reachteam'],
    'Self-Efficacy\n(Goals)':   ['confidentgoals'],
    'Self-Efficacy\n(Resources)':['confidentresources'],
    'Self-Efficacy\n(Pain)':    ['confidentmanagepain'],
    'Comprehension\n(avg)':     ['understandmaterials','understandmail',
                                  'understandcalls','understandactionplan'],
    'Helpfulness\n(avg)':       ['helpfulmaterials','helpfulmailed',
                                  'helpfulcalls','helpfulactionplan'],
}

subgroups = {
    'Female':               survey_eff2['sex_numerical'] == 1,
    'Male':                 survey_eff2['sex_numerical'] == 0,
    'Hispanic/Latino':      survey_eff2['ethnicity_hispanic'] == 1,
    'Non-Hispanic':         survey_eff2['ethnicity_hispanic'] == 0,
    'Rural (RUCA 7-10)':    survey_eff2['rucacat'].isin([3, 4]),
    'Urban/Large Rural':    survey_eff2['rucacat'].isin([1, 2]),
    'Qualifying: Rural':    survey_eff2['qualifyingfeature_num'] == 1,
    'Qualifying: Latinx':   survey_eff2['qualifyingfeature_num'] == 2,
}

def domain_mean(mask, cols):
    vals = survey_eff2.loc[mask, cols].values.flatten()
    vals = vals[~np.isnan(vals.astype(float))]
    return round(np.mean(vals), 2) if len(vals) else np.nan

hm_data = {}
for sg_name, mask in subgroups.items():
    n_sg = mask.sum()
    row_label = f'{sg_name} (n={n_sg})'
    hm_data[row_label] = {
        dom: domain_mean(mask, cols)
        for dom, cols in domain_cols.items()
    }

hm_sg = pd.DataFrame(hm_data).T

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(
    hm_sg,
    ax=ax,
    vmin=1, vmax=5,
    cmap='RdYlGn',
    annot=True,
    fmt='.2f',
    annot_kws={'size': 10},
    linewidths=0.6,
    linecolor='white',
    cbar_kws={'label': 'Mean score (1–5)', 'shrink': 0.7}
)
ax.set_title('Mean Outcome Scores by Demographic Subgroup', fontsize=13, fontweight='bold', pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

# Dividers between subgroup pairs
for line_pos in [2, 4, 6]:
    ax.axhline(line_pos, color='#94A3B8', linewidth=1.5, linestyle='--')

plt.tight_layout()
plt.savefig('fig_heatmap_subgroup.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashed lines separate subgroup pairs (sex | ethnicity | rurality | qualifying feature)')